# ML-04 — Search Intelligence Data Contract

**Lane:** Refresh / Content Opportunity Scoring  
**Intern:** Varshan M  

Four things, in order:
1. The contract — 5 plain-words answers
2. Prove three facts with three real queries on a mid-panel month (`month=2026-03`)
3. Five features + the deliberate-leak trap
4. One named limitation of this slice

> Skills loaded per `skills/README.md`: `writing-data-contracts/SKILL.md` + `flyrank/flyrank-data/SKILL.md`

**Warning before iterating:** the `_sample` table is the final month (June 2026) — using it to develop label logic leaks the outcome window. All queries here use `month=2026-03` as the iteration partition.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, getpass

# In Colab: store your READ token as a Secret named HF_TOKEN (the key panel).
# Never paste the token into a code cell — this repo is public.
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

print('Token loaded:', HF_TOKEN[:8] + '...' if HF_TOKEN else 'MISSING')

Token loaded: hf_kIILB...


In [3]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'

# Table references — use partitioned path for the fact table
DAILY  = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet', hive_partitioning=true)"
# Mid-panel month for all iteration queries (never the final/sample month)
MARCH  = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_C  = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CT = f"read_parquet('{REL}/dim_content.parquet')"

print('DuckDB connection ready. Iteration partition: month=2026-03')

DuckDB connection ready. Iteration partition: month=2026-03


## 1. Unit of analysis + time window

The contract, in 5 plain-words answers:

| Question | Answer |
|---|---|
| **What does one row mean?** | One content page (article) for one client, on one calendar day — a `report_date × client_hash_id × content_hash_id` triplet in `fact_content_daily_performance`. For feature building, we aggregate this to one row per content item per 30-day window. |
| **Which table(s)?** | Primary: `fact_content_daily_performance` (daily search + engagement signals). Context joins: `dim_clients` (per-client history coverage) and `dim_content` (content metadata, keyword context). |
| **Which time window?** | Feature window: the 60 days **before** a chosen cut-date (`impressions_prev30` = days −60 to −31; `impressions_last30` = days −30 to 0). Label window: the 30 days immediately after the cut-date. For this notebook we iterate on **`month=2026-03`** as the feature period; the final month (`2026-06`) is sealed as a test partition and never used for label development. |
| **What would you predict / rank (label or proxy)?** | **Proxy label — `is_declining`**: 1 when `impressions_last30 < 0.80 × impressions_prev30` (impressions fell >20% in the most recent 30 days vs the prior 30 days). This is an observed outcome derived from measured GSC impression counts, not a product rule. It is a *proxy* because drops can reflect seasonality or algorithm shifts, not just stale content. |
| **One thing you deliberately exclude** | `gsc_avg_position` from the **same 30-day label window** is excluded from features: position in the outcome period contains information about whether a page declined (declining pages lose position too), making it a leakage risk. Only `gsc_avg_position` from the *feature* (prev30) window is safe. |

**Field classification:**

| Field | Bucket | Why |
|---|---|---|
| `gsc_impressions` (prev30 window sum) | Feature | Observed before decision point |
| `gsc_clicks` (prev30 window sum) | Feature | Observed before decision point |
| `gsc_avg_position` (prev30 window avg) | Feature | Average position in feature window only |
| `ga4_sessions` (prev30 window sum) | Feature | Engagement signal, observed before prediction |
| `days_with_gsc_data` (prev30 count) | Feature | Data density / consistency signal |
| `gsc_impressions` (last30 window) | **Label source** | Used to compute `is_declining` — never a feature |
| `client_hash_id`, `content_hash_id` | Context | Join keys; pseudonymous IDs, never model inputs |
| `ga4_data_available` | Context | Filter flag — rows with `FALSE` have zero-filled GA4 columns |
| `gsc_avg_position` (last30 window) | **Excluded** | Future information: position during the label window correlates directly with decline |
| `ga4_sessions_ai` | Excluded for now | Extremely sparse (30K rows in 79M) — not a reliable signal for this lane |

**Output:** A ranked refresh queue — one row per content item, ordered by predicted probability of decline, with a reason code for the editor.

## 2. Fields: feature / label / context / excluded

*(Covered in the contract table above — the three queries below verify every claim.)*

## 3. Verify it with queries (grain, counts, availability)

**Three queries. Every contract claim gets a query next to it.**

- **Query A:** Grain check — confirm the grain is `report_date × client_hash_id × content_hash_id` (zero duplicate triplets).
- **Query B:** Row count and date span — confirm the March 2026 partition's size and date window.
- **Query C:** Availability check — how many rows survive the `ga4_data_available IS TRUE` filter (GA4 columns are zero-filled before a client's GA4 start; using those zeros as "no engagement" is wrong).

In [4]:
# ── Query A: Grain check ───────────────────────────────────────────────────────
# Contract claim: one row = one report_date × client_hash_id × content_hash_id
# Zero rows back means the grain holds (no duplicates).

print("=== Query A: Grain check — month=2026-03 ===")
print("SQL: SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c")
print("     FROM march_partition")
print("     GROUP BY 1,2,3 HAVING c > 1 LIMIT 5")
print()

result_a = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MARCH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print(f"Duplicate-triplet rows returned: {len(result_a)}")
if len(result_a) == 0:
    print("PASS: grain holds — no duplicate report_date × client × content rows.")
else:
    print("FAIL: duplicates found — investigate before modeling.")
    print(result_a)

=== Query A: Grain check — month=2026-03 ===
SQL: SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
     FROM march_partition
     GROUP BY 1,2,3 HAVING c > 1 LIMIT 5



Duplicate-triplet rows returned: 0
PASS: grain holds — no duplicate report_date × client × content rows.


In [5]:
# ── Query B: Row count and date span ──────────────────────────────────────────
# Contract claim: month=2026-03 is a ~31-day window, one day per calendar date.

print("=== Query B: Row count + date span — month=2026-03 ===")
print()

result_b = con.sql(f"""
    SELECT
        COUNT(*)                    AS total_rows,
        COUNT(DISTINCT report_date) AS distinct_days,
        MIN(report_date)            AS min_date,
        MAX(report_date)            AS max_date,
        COUNT(DISTINCT client_hash_id)  AS distinct_clients,
        COUNT(DISTINCT content_hash_id) AS distinct_content_items
    FROM {MARCH}
""").df()

print(result_b.to_string(index=False))
print()
print("Interpretation: total_rows / distinct_content_items / distinct_clients")
tr = result_b['total_rows'].iloc[0]
dc = result_b['distinct_clients'].iloc[0]
di = result_b['distinct_content_items'].iloc[0]
print(f"  {tr:,} daily rows across {di:,} content items and {dc} clients")

=== Query B: Row count + date span — month=2026-03 ===



 total_rows  distinct_days   min_date   max_date  distinct_clients  distinct_content_items
    9841378             31 2026-03-01 2026-03-31                55                  331437

Interpretation: total_rows / distinct_content_items / distinct_clients
  9,841,378 daily rows across 331,437 content items and 55 clients


In [6]:
# ── Query C: Availability check with IS TRUE ───────────────────────────────────
# Contract claim: rows before a client's ga4_data_start have ga4_data_available = FALSE
#   and their GA4 columns are zero-filled — they look like "no engagement" but are really
#   "no tracking yet". Filter with IS TRUE before using ANY GA4 column.

print("=== Query C: GA4 availability — month=2026-03 ===")
print("Checking ga4_data_available IS TRUE vs total rows")
print()

result_c = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE  THEN 1 ELSE 0 END) AS ga4_available_rows,
        SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_unavailable_rows,
        SUM(CASE WHEN ga4_data_available IS NULL  THEN 1 ELSE 0 END) AS ga4_null_rows,
        ROUND(
            100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*),
            1
        ) AS pct_ga4_available
    FROM {MARCH}
""").df()

print(result_c.to_string(index=False))
print()
avail = result_c['ga4_available_rows'].iloc[0]
total = result_c['total_rows'].iloc[0]
pct   = result_c['pct_ga4_available'].iloc[0]
print(f"Rows safe to use GA4 signals: {avail:,} / {total:,}  ({pct}%)")
print(f"Rows to EXCLUDE (zero-filled GA4): {total - avail:,}")
print()
print("ACTION: always filter `WHERE ga4_data_available IS TRUE` before using")
print("ga4_sessions, ga4_sessions_engaged, ga4_sessions_ai, ga4_page_views.")

=== Query C: GA4 availability — month=2026-03 ===
Checking ga4_data_available IS TRUE vs total rows



 total_rows  ga4_available_rows  ga4_unavailable_rows  ga4_null_rows  pct_ga4_available
    9841378            413966.0             6408671.0      3018741.0                4.2

Rows safe to use GA4 signals: 413,966.0 / 9,841,378  (4.2%)
Rows to EXCLUDE (zero-filled GA4): 9,427,412.0

ACTION: always filter `WHERE ga4_data_available IS TRUE` before using
ga4_sessions, ga4_sessions_engaged, ga4_sessions_ai, ga4_page_views.


## 3 (continued) — Five features and the deliberate-leak trap

### Five features, with one "available when?" line each

All five are built from the **prev30 window** (days −60 to −31 before the cut-date).  
None of them touch the label window (days −30 to 0).

| Feature | Definition | Available when? |
|---|---|---|
| `imp_prev30` | Sum of `gsc_impressions` in the prior 30-day window | Knowable at the decision moment because it is measured before the label period begins |
| `clk_prev30` | Sum of `gsc_clicks` in the prior 30-day window | Knowable at the decision moment because clicks are recorded by Google Search Console before the outcome window |
| `pos_prev30` | Average `gsc_avg_position` in the prior 30-day window | Knowable at the decision moment because it is the page's ranking *before* any decline we're trying to predict |
| `sess_prev30` | Sum of `ga4_sessions` in the prior 30-day window (GA4-available rows only) | Knowable at the decision moment because GA4 session data is available once tracking is active; filtered to `ga4_data_available IS TRUE` rows only |
| `days_with_imp_prev30` | Count of days in the prior 30-day window where `gsc_impressions > 0` | Knowable at the decision moment because it describes past data consistency — a page with sparse impression days is noisier than one with consistent coverage |

### The deliberate-leak trap (perform it, then remove it)

We will:
1. Build the honest feature frame from prev30 signals.
2. Add `imp_last30` (from the label window) as a feature on purpose — this is the trap.
3. Watch the score jump toward perfect.
4. Remove it and keep only the honest number.

In [7]:
# ── Build the feature frame from month=2026-03 ────────────────────────────────
# Feature window:  2026-02-01 to 2026-03-01  (prev30 = days before mid-month cut)
# Label window:    2026-03-01 to 2026-03-31  (last30 = the March outcome)
# Cut-date: 2026-03-01  (features before, label after)

print("Building feature frame from month=2026-03 partition...")
print("Feature window (prev30): 2026-02-01 to 2026-02-28")
print("Label window  (last30):  2026-03-01 to 2026-03-31")
print()

# We pull TWO months of data to build prev30 and last30 windows cleanly.
# February = feature window; March = label window.
FEB = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-02/*.parquet')"

frame = con.sql(f"""
    WITH feb AS (
        -- prev30: February 2026 (feature window)
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions)                                             AS imp_prev30,
            SUM(gsc_clicks)                                                  AS clk_prev30,
            AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END)   AS pos_prev30,
            SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions ELSE 0 END) AS sess_prev30,
            COUNT(CASE WHEN gsc_impressions > 0 THEN 1 END)                 AS days_with_imp_prev30
        FROM {FEB}
        GROUP BY 1, 2
        HAVING imp_prev30 >= 50  -- minimum volume: exclude noise-level pages
    ),
    mar AS (
        -- last30: March 2026 (label window — used ONLY for the label, not features)
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS imp_last30
        FROM {MARCH}
        GROUP BY 1, 2
    )
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        f.imp_prev30,
        f.clk_prev30,
        f.pos_prev30,
        f.sess_prev30,
        f.days_with_imp_prev30,
        m.imp_last30,
        -- Label: 1 if impressions fell >20% in March vs February
        CASE WHEN m.imp_last30 < 0.80 * f.imp_prev30 THEN 1 ELSE 0 END AS is_declining
    FROM feb f
    LEFT JOIN mar m USING (client_hash_id, content_hash_id)
""").df()

print(f"Feature frame shape: {frame.shape}")
print(f"Declining pages: {frame['is_declining'].sum():,} / {len(frame):,}  "
      f"({frame['is_declining'].mean()*100:.1f}%)")
print()
print("Sample (5 rows):")
frame.head()

Building feature frame from month=2026-03 partition...
Feature window (prev30): 2026-02-01 to 2026-02-28
Label window  (last30):  2026-03-01 to 2026-03-31



Feature frame shape: (93654, 9)
Declining pages: 16,905 / 93,654  (18.1%)

Sample (5 rows):


,client_hash_id,content_hash_id,imp_prev30,clk_prev30,pos_prev30,sess_prev30,days_with_imp_prev30,imp_last30,is_declining
0,client_73cda7b4e4f265ea,content_f12c512e61711572,490.0,0.0,4.563080,0.0,28,716.0,0
1,client_73cda7b4e4f265ea,content_1982151c501b28f1,949.0,0.0,4.337366,0.0,28,1525.0,0
2,client_73cda7b4e4f265ea,content_c059ab287c00213f,40383.0,103.0,4.144201,0.0,28,36865.0,0
3,client_73cda7b4e4f265ea,content_62ae9fc575dc5a9c,5299.0,5.0,5.711190,0.0,28,8096.0,0
4,client_73cda7b4e4f265ea,content_1d00306816b67ea2,68.0,0.0,20.357465,0.0,19,492.0,0


In [8]:
# ── Quick model: HONEST features only ────────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

HONEST_FEATURES = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'sess_prev30', 'days_with_imp_prev30']

clean = frame.dropna(subset=HONEST_FEATURES + ['is_declining'])
X_honest = clean[HONEST_FEATURES]
y = clean['is_declining']

X_tr, X_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42, stratify=y)
rf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
auc_honest = roc_auc_score(y_te, rf_honest.predict_proba(X_te)[:, 1])

print(f"=== HONEST model (5 prev30 features) ===")
print(f"Base rate (always-positive): {y_te.mean():.3f}")
print(f"ROC-AUC (honest):  {auc_honest:.3f}")
print()
print("This is the number we keep. Onto the trap...")

=== HONEST model (5 prev30 features) ===
Base rate (always-positive): 0.180
ROC-AUC (honest):  0.624

This is the number we keep. Onto the trap...


In [9]:
# ── DELIBERATE LEAK: add imp_last30 as a feature ─────────────────────────────
# This is the trap everyone falls into — using the label-window metric as a feature.
# imp_last30 IS the numerator of the label formula:
#   is_declining = (imp_last30 < 0.8 * imp_prev30)
# A model trained with imp_last30 as a feature learns to reproduce the label perfectly.

LEAKY_FEATURES = HONEST_FEATURES + ['imp_last30']  # <-- the leak

clean_lk = frame.dropna(subset=LEAKY_FEATURES + ['is_declining'])
X_leaky = clean_lk[LEAKY_FEATURES]
y_lk = clean_lk['is_declining']

X_tr_lk, X_te_lk, y_tr_lk, y_te_lk = train_test_split(
    X_leaky, y_lk, test_size=0.25, random_state=42, stratify=y_lk
)
rf_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1).fit(X_tr_lk, y_tr_lk)
auc_leaky = roc_auc_score(y_te_lk, rf_leaky.predict_proba(X_te_lk)[:, 1])

print(f"=== LEAKY model (5 honest features + imp_last30) ===")
print(f"ROC-AUC (leaky):   {auc_leaky:.3f}   <-- suspiciously high")
print()
print(f"Gap vs honest model: {auc_leaky - auc_honest:+.3f}")
print()
print("WHY this happened:")
print("  imp_last30 is directly part of the label formula:")
print("    is_declining = (imp_last30 < 0.8 * imp_prev30)")
print("  Giving the model imp_last30 is giving it the answer sheet.")
print("  The model learns the rule, not the world. Zero generalization.")

# Feature importances reveal the smoking gun
importances = pd.Series(
    rf_leaky.feature_importances_, index=LEAKY_FEATURES
).sort_values(ascending=False)
print()
print("Feature importances in the leaky model (imp_last30 dominates):")
print(importances.to_string())

=== LEAKY model (5 honest features + imp_last30) ===
ROC-AUC (leaky):   0.999   <-- suspiciously high

Gap vs honest model: +0.376

WHY this happened:
  imp_last30 is directly part of the label formula:
    is_declining = (imp_last30 < 0.8 * imp_prev30)
  Giving the model imp_last30 is giving it the answer sheet.
  The model learns the rule, not the world. Zero generalization.

Feature importances in the leaky model (imp_last30 dominates):
imp_last30              0.566624
imp_prev30              0.325570
pos_prev30              0.051222
days_with_imp_prev30    0.023830
clk_prev30              0.021060
sess_prev30             0.011693


In [10]:
# ── Remove the leak, keep the honest number ──────────────────────────────────
# imp_last30 is deleted from features here and is NEVER used again as a model input.
# It is only used to define the label (which happens before training).

# The label itself is fine — it is computed from imp_last30, but the model never
# sees imp_last30 as a column during training. This is the correct pattern.

print("=== Summary of the trap ===")
print(f"Honest ROC-AUC  (prev30 features only):       {auc_honest:.3f}")
print(f"Leaky  ROC-AUC  (prev30 + imp_last30 leaked): {auc_leaky:.3f}")
print()
print("imp_last30 is now REMOVED from all feature sets.")
print("The honest number is what we report: {:.3f}".format(auc_honest))
print()
print("Lesson: a jump toward perfect AUC is a red flag, not a win.")
print("Always check: does any feature column derive from or overlap the label window?")

# Show the clean five-feature frame for downstream use
print()
print("Five safe features — ready for downstream modeling:")
print(clean[HONEST_FEATURES + ['is_declining']].describe().round(2).to_string())

=== Summary of the trap ===
Honest ROC-AUC  (prev30 features only):       0.624
Leaky  ROC-AUC  (prev30 + imp_last30 leaked): 0.999

imp_last30 is now REMOVED from all feature sets.
The honest number is what we report: 0.624

Lesson: a jump toward perfect AUC is a red flag, not a win.
Always check: does any feature column derive from or overlap the label window?

Five safe features — ready for downstream modeling:
       imp_prev30  clk_prev30  pos_prev30  sess_prev30  days_with_imp_prev30  is_declining
count    93649.00    93649.00    93649.00     93649.00              93649.00      93649.00
mean      1915.33        6.22       12.24         3.42                 24.02          0.18
std       5018.55       27.78       12.03        22.63                  6.61          0.38
min         50.00        0.00        0.06         0.00                  1.00          0.00
25%        164.00        0.00        4.73         0.00                 22.00          0.00
50%        507.00        1.00       

## 4. Data limits

**Named limitation of this slice:**

**Unbalanced panel — per-client history depth varies wildly.** Some clients have 17 months of GSC data (since January 2025); others joined the platform recently and have fewer than 3 months. A feature like `imp_prev30` therefore measures very different things for a long-running client (stable average) vs a new client (possibly onboarding noise). Any model trained without checking `dim_clients.gsc_data_start` per client risks mixing those two situations — and misinterpreting a new client's low-volume first month as "declining" when it was just starting.

**What this data cannot tell you:**
- Whether a refresh *caused* a recovery (no experiment design; observational only).
- Whether a drop is seasonal vs structural — with < 12 months of history for most clients, year-over-year comparison is not always possible.
- What the actual content says — all text fields were stripped from the warehouse release; we have only safe observable signals.
- Whether GA4 zeros are real engagement zeros or just missing tracking — rows with `ga4_data_available = FALSE` must always be filtered out, not treated as low-engagement.

In [11]:
# Demonstrate the panel imbalance from dim_clients
print("=== Panel imbalance: per-client GSC history depth ===")

clients = con.sql(f"""
    SELECT
        access_profile,
        gsc_data_start,
        ga4_data_start,
        DATE_DIFF('day', gsc_data_start, DATE '2026-06-30') AS gsc_history_days
    FROM {DIM_C}
    WHERE gsc_data_start IS NOT NULL
    ORDER BY gsc_history_days DESC
""").df()

print(f"Clients with GSC data: {len(clients)}")
print()
print("Distribution of GSC history depth (days):")
print(clients['gsc_history_days'].describe().round(0).to_string())
print()
print("Clients with < 90 days of GSC history:", (clients['gsc_history_days'] < 90).sum())
print("Clients with >= 365 days of GSC history:", (clients['gsc_history_days'] >= 365).sum())
print()
print("LIMITATION: model trained on all clients mixes deep-history stable signals")
print("with shallow-history noisy signals. Always filter on gsc_data_start BEFORE")
print("defining any time window, or stratify by client history depth.")

=== Panel imbalance: per-client GSC history depth ===


Clients with GSC data: 67

Distribution of GSC history depth (days):
count     67.0
mean     225.0
std      125.0
min       28.0
25%      131.0
50%      237.0
75%      279.0
max      519.0

Clients with < 90 days of GSC history: 10
Clients with >= 365 days of GSC history: 9

LIMITATION: model trained on all clients mixes deep-history stable signals
with shallow-history noisy signals. Always filter on gsc_data_start BEFORE
defining any time window, or stratify by client history depth.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Three verification queries shown with output (grain check, row count/date span, availability with IS TRUE)
- [x] Five features listed with one "available when?" line each
- [x] Deliberate-leak experiment performed, scored, and removed — honest number kept
- [x] One named limitation stated and verified with a query
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.